In [30]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

In [31]:
def get_stats(ids):
    counts = {}

    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1

    return counts


def merge(ids, pair, new_id):
    new_ids = []
    i = 0

    while i < len(ids):
        if (
            i + 1 < len(ids)
            and ids[i] == pair[0]
            and ids[i + 1] == pair[1]
        ):
            new_ids.append(new_id)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1

    return new_ids


# Build token ID -> byte sequence mapping.
def build_vocab(merges):
    vocab = {
        token_id: bytes([token_id])
        for token_id in range(256)
    }

    for pair, new_id in merges.items():
        vocab[new_id] = (
            vocab[pair[0]] + vocab[pair[1]]
        )

    return vocab


def encode(text, merges):
    ids = list(str(text).encode("utf-8"))

    # Earlier learned merges have higher priority.
    merge_ranks = {
        pair: rank
        for rank, pair in enumerate(merges.keys())
    }

    while len(ids) >= 2:
        stats = get_stats(ids)

        # Find pairs that exist in the learned merges.
        valid_pairs = [
            pair
            for pair in stats
            if pair in merges
        ]

        if not valid_pairs:
            break

        # Select the earliest learned pair.
        best_pair = min(
            valid_pairs,
            key=lambda pair: merge_ranks[pair],
        )

        ids = merge(
            ids,
            best_pair,
            merges[best_pair],
        )

    return ids


def decode(ids, vocab):
    byte_sequence = b"".join(
        vocab[token_id]
        for token_id in ids
    )

    return byte_sequence.decode("utf-8")

In [32]:
def load_tokenizer(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Tokenizer file not found: {path.resolve()}"
        )

    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    # Restore ordered BPE merge rules.
    merges = {
        (first_id, second_id): new_id
        for first_id, second_id, new_id in data["merges"]
    }

    vocab = build_vocab(merges)

    return merges, vocab, data["vocab_size"]

In [33]:
tokenizer_path = Path("tokenizer/tokenizer/tokenizer.json")
if not tokenizer_path.exists():
    tokenizer_path = Path("..") / tokenizer_path

merges, vocab, vocab_size = load_tokenizer(tokenizer_path)

print("Tokenizer loaded")
print("Vocabulary size:", vocab_size)
print("Number of merges:", len(merges))

Tokenizer loaded
Vocabulary size: 1000
Number of merges: 744


In [34]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder


def build_fast_tokenizer(merges, vocab):
    byte_values = (
        list(range(ord("!"), ord("~") + 1))
        + list(range(ord("¡"), ord("¬") + 1))
        + list(range(ord("®"), ord("ÿ") + 1))
    )

    unicode_values = byte_values.copy()
    extra_index = 0

    for byte_value in range(256):
        if byte_value not in byte_values:
            byte_values.append(byte_value)
            unicode_values.append(256 + extra_index)
            extra_index += 1

    byte_encoder = {
        byte_value: chr(unicode_value)
        for byte_value, unicode_value
        in zip(byte_values, unicode_values)
    }

    def bytes_to_token_string(byte_sequence):
        return "".join(
            byte_encoder[byte_value]
            for byte_value in byte_sequence
        )

    fast_vocab = {
        bytes_to_token_string(byte_sequence): token_id
        for token_id, byte_sequence in vocab.items()
    }

    fast_merges = [
        (
            bytes_to_token_string(vocab[first_id]),
            bytes_to_token_string(vocab[second_id]),
        )
        for first_id, second_id in merges
    ]

    tokenizer = Tokenizer(
        BPE(
            vocab=fast_vocab,
            merges=fast_merges,
        )
    )

    tokenizer.pre_tokenizer = ByteLevel(
        add_prefix_space=False,
        use_regex=False,
    )

    tokenizer.decoder = ByteLevelDecoder()

    return tokenizer


fast_tokenizer = build_fast_tokenizer(
    merges=merges,
    vocab=vocab,
)

print("Fast tokenizer created")
print("Vocabulary size:", fast_tokenizer.get_vocab_size())

Fast tokenizer created
Vocabulary size: 1000


In [35]:
print("Vocabulary size:", vocab_size)
print("Base byte tokens:", 256)
print("Learned BPE tokens:", len(merges))
print("Calculated size:", 256 + len(merges))

# Shared model configuration
vocab_size = 1000
context_length = 512
embedding_dim = 96
num_heads = 6
num_layers = 4
batch_size = 32

Vocabulary size: 1000
Base byte tokens: 256
Learned BPE tokens: 744
Calculated size: 1000


In [36]:
sample = "This is a tokenizer test."

token_ids = encode(sample, merges)
reconstructed = decode(token_ids, vocab)

print("Token IDs:", token_ids)
print("Decoded:", reconstructed)
print("Exact match:", sample == reconstructed)

Token IDs: [84, 398, 445, 289, 313, 107, 276, 105, 122, 292, 938, 304, 46]
Decoded: This is a tokenizer test.
Exact match: True


In [37]:
sample = "This is a tokenizer test."

original_ids = encode(sample, merges)
fast_ids = fast_tokenizer.encode(sample).ids

reconstructed = fast_tokenizer.decode(fast_ids)

print("Original IDs:", original_ids)
print("Fast IDs:", fast_ids)
print("Same token IDs:", original_ids == fast_ids)
print("Decoded:", reconstructed)
print("Exact match:", sample == reconstructed)

Original IDs: [84, 398, 445, 289, 313, 107, 276, 105, 122, 292, 938, 304, 46]
Fast IDs: [84, 398, 445, 289, 313, 107, 276, 105, 122, 292, 938, 304, 46]
Same token IDs: True
Decoded: This is a tokenizer test.
Exact match: True


In [38]:
data_path = Path("tinystories_100mb.jsonl")
if not data_path.exists():
    data_path = Path("..") / data_path

all_data = pd.read_json(data_path, lines=True)

text = all_data["text"].dropna().astype(str).str.cat(sep="\n") + "\n"

In [39]:
len(text)

101855429

In [40]:
### encoding entire data 
import time

start_time = time.perf_counter()

token_ids = fast_tokenizer.encode(text).ids

encoding_time = time.perf_counter() - start_time

print(f"Characters: {len(text):,}")
print(f"Tokens: {len(token_ids):,}")
print(f"Encoding time: {encoding_time:.2f} seconds")

Characters: 101,855,429
Tokens: 31,951,061
Encoding time: 102.67 seconds


- All Set for transformer 

In [41]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [42]:
class LanguageModelDataset(Dataset):
    def __init__(self, token_ids, context_length):
        if len(token_ids) <= context_length:
            raise ValueError(
                "The number of tokens must be greater "
                "than context_length."
            )

        self.token_ids = torch.tensor(
            token_ids,
            dtype=torch.long,
        )

        self.context_length = context_length

    def __len__(self):
        return len(self.token_ids) - self.context_length

    def __getitem__(self, index):
        x = self.token_ids[
            index : index + self.context_length
        ]

        y = self.token_ids[
            index + 1 : index + self.context_length + 1
        ]

        return x, y

In [43]:
split_index = int(len(token_ids) * 0.8)

train_token_ids = token_ids[:split_index]
validation_token_ids = token_ids[split_index:]

if len(train_token_ids) <= context_length:
    raise ValueError("Not enough training tokens for context_length")

if len(validation_token_ids) <= context_length:
    raise ValueError("Not enough validation tokens for context_length")

train_dataset = LanguageModelDataset(
    token_ids=train_token_ids,
    context_length=context_length,
)

validation_dataset = LanguageModelDataset(
    token_ids=validation_token_ids,
    context_length=context_length,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=False,
)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(validation_dataset))
print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

Training examples: 25560336
Validation examples: 6389701
Training batches: 798761
Validation batches: 199679


In [44]:
X_batch, y_batch = next(iter(train_loader))

print("X batch shape:", X_batch.shape)
print("y batch shape:", y_batch.shape)

X batch shape: torch.Size([32, 512])
y batch shape: torch.Size([32, 512])


In [45]:
class EmbeddingLayer(nn.Module):

    def __init__(self, vocab_size, context_length, embedding_dim):
        super().__init__()

        # Token Embedding Matrix
        # Shape: [vocab_size, embedding_dim]
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim
        )

        # Positional Embedding Matrix
        # Shape: [context_length, embedding_dim]
        self.position_embedding = nn.Embedding(
            num_embeddings=context_length,
            embedding_dim=embedding_dim
        )

    def forward(self, input_ids):

        # input_ids shape:
        # [batch_size, sequence_length]

        batch_size, sequence_length = input_ids.shape

        # Create position IDs
        positions = torch.arange(
            sequence_length,
            device=input_ids.device
        )

        # Convert token IDs into vectors
        token_embeddings = self.token_embedding(input_ids)

        # Convert position IDs into vectors
        position_embeddings = self.position_embedding(positions)

        # Add token information + position information
        embeddings = token_embeddings + position_embeddings

        return embeddings

In [46]:
embedding_layer = EmbeddingLayer(
    vocab_size=vocab_size,
    context_length=context_length,
    embedding_dim=embedding_dim
)

In [47]:
embeddings = embedding_layer(X_batch)

print("X:", X_batch.shape)
print("y:", y_batch.shape)
print("Embeddings:", embeddings.shape)

X: torch.Size([32, 512])
y: torch.Size([32, 512])
Embeddings: torch.Size([32, 512, 96])


In [48]:
print("Token embedding matrix:",
      embedding_layer.token_embedding.weight.shape)

print("Position embedding matrix:",
      embedding_layer.position_embedding.weight.shape)

print("Output embeddings:",
      embeddings.shape)

Token embedding matrix: torch.Size([1000, 96])
Position embedding matrix: torch.Size([512, 96])
Output embeddings: torch.Size([32, 512, 96])


In [49]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, context_length):
        super().__init__()

        self.d_model = d_model

        # Learnable projections for Query, Key, Value
        self.query = nn.Linear(d_model, d_model, bias=False)
        self.key   = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(d_model, d_model, bias=False)

        # Causal mask
        mask = torch.tril(torch.ones(context_length, context_length))

        # register_buffer means:
        # - it is stored with the model
        # - moved automatically to CPU/GPU
        # - but is NOT trained
        self.register_buffer("mask", mask)

    def forward(self, x):
        """
        x shape:
        [batch_size, sequence_length, d_model]
        """

        B, T, C = x.shape

        # 1. Generate Query, Key, Value
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        # Shapes:
        # Q: [B, T, C]
        # K: [B, T, C]
        # V: [B, T, C]

        # 2. Compute attention scores
        attention_scores = Q @ K.transpose(-2, -1)

        # shape:
        # [B, T, T]

        # 3. Scale attention scores
        attention_scores = attention_scores / (C ** 0.5)

        # 4. Apply causal mask
        attention_scores = attention_scores.masked_fill(
            self.mask[:T, :T] == 0,
            float("-inf")
        )

        # 5. Convert scores into probabilities
        attention_weights = F.softmax(attention_scores, dim=-1)

        # 6. Weighted sum of Value vectors
        output = attention_weights @ V

        # output shape:
        # [B, T, C]

        return output

In [50]:
demo_batch_size = 2
demo_sequence_length = 5

x = torch.randn(
    demo_batch_size,
    demo_sequence_length,
    embedding_dim
)

attention = CausalSelfAttention(
    d_model=embedding_dim,
    context_length=context_length
)

output = attention(x)

print("Input :", x.shape)
print("Output:", output.shape)

Input : torch.Size([2, 5, 96])
Output: torch.Size([2, 5, 96])


In [51]:
class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, context_length):
        super().__init__()

        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.query = nn.Linear(d_model, d_model, bias=False)
        self.key = nn.Linear(d_model, d_model, bias=False)
        self.value = nn.Linear(d_model, d_model, bias=False)

        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        mask = torch.tril(
            torch.ones(context_length, context_length)
        )

        self.register_buffer(
            "mask",
            mask.view(1, 1, context_length, context_length)
        )

    def forward(self, x):
        B, T, C = x.shape

        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        Q = Q.view(B, T, self.n_heads, self.head_dim)
        K = K.view(B, T, self.n_heads, self.head_dim)
        V = V.view(B, T, self.n_heads, self.head_dim)

        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        scores = Q @ K.transpose(-2, -1)

        scores = scores / (self.head_dim ** 0.5)

        scores = scores.masked_fill(
            self.mask[:, :, :T, :T] == 0,
            float("-inf")
        )

        attention_weights = F.softmax(scores, dim=-1)

        out = attention_weights @ V

        out = out.transpose(1, 2)
        out = out.contiguous().view(B, T, C)

        out = self.out_proj(out)

        return out

In [52]:
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.fc1 = nn.Linear(d_model, 4 * d_model)
        self.fc2 = nn.Linear(4 * d_model, d_model)

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)

        return x

class LayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        return self.norm(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, context_length):
        super().__init__()

        self.ln1 = LayerNorm(d_model)

        self.attention = MultiHeadCausalSelfAttention(
            d_model=d_model,
            n_heads=n_heads,
            context_length=context_length
        )

        self.ln2 = LayerNorm(d_model)

        self.ffn = FeedForward(d_model)

    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        x = x + self.ffn(self.ln2(x))

        return x

In [53]:
transformer_blocks = nn.ModuleList([
    TransformerBlock(
        d_model=embedding_dim,
        n_heads=num_heads,
        context_length=context_length
    )
    for _ in range(num_layers)
])

x = torch.randn(2, 10, embedding_dim)

output = x
for block in transformer_blocks:
    output = block(output)

print("Transformer layers:", len(transformer_blocks))
print("Input shape :", x.shape)
print("Output shape:", output.shape)

Transformer layers: 4
Input shape : torch.Size([2, 10, 96])
Output shape: torch.Size([2, 10, 96])


In [54]:
class SmallLanguageModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        context_length,
        embedding_dim,
        num_heads,
        num_layers,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.context_length = context_length

        self.embedding = EmbeddingLayer(
            vocab_size=vocab_size,
            context_length=context_length,
            embedding_dim=embedding_dim,
        )

        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(
                d_model=embedding_dim,
                n_heads=num_heads,
                context_length=context_length,
            )
            for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(embedding_dim)

        self.lm_head = nn.Linear(
            embedding_dim,
            vocab_size,
            bias=False,
        )

        # Share input embedding and output weights.
        self.lm_head.weight = self.embedding.token_embedding.weight

    def forward(self, input_ids):
        if input_ids.ndim != 2:
            raise ValueError(
                "input_ids must have shape "
                "[batch_size, sequence_length]"
            )

        if input_ids.size(1) > self.context_length:
            raise ValueError(
                f"Sequence length {input_ids.size(1)} exceeds "
                f"context length {self.context_length}"
            )

        x = self.embedding(input_ids)

        for block in self.transformer_blocks:
            x = block(x)

        x = self.final_norm(x)

        # Shape: [batch, sequence, vocabulary]
        logits = self.lm_head(x)

        return logits

In [55]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = SmallLanguageModel(
    vocab_size=vocab_size,
    context_length=context_length,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers,
).to(device)

print("Device:", device)

number_of_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(f"Parameters: {number_of_parameters:,}")

Device: cuda
Parameters: 591,168


In [56]:
@torch.no_grad()
def evaluate(model, data_loader, loss_function, device):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for input_ids, targets in data_loader:
        input_ids = input_ids.to(device)
        targets = targets.to(device)

        logits = model(input_ids)

        loss = loss_function(
            logits.reshape(-1, model.vocab_size),
            targets.reshape(-1),
        )

        number_of_tokens = targets.numel()

        total_loss += loss.item() * number_of_tokens
        total_tokens += number_of_tokens

    return total_loss / total_tokens

In [63]:
import copy
from itertools import islice
from pathlib import Path

import torch
import torch.nn as nn


# Training configuration
learning_rate = 3e-4
num_epochs = 1

# Early-stopping configuration
early_stopping_patience = 10
minimum_improvement = 0.001

# Save progress approximately every 250 batches.
checkpoint_frequency = 250

# Find the same checkpoint folder whether Jupyter starts
# in model/, the repository root, or its parent folder.
checkpoint_directory_candidates = (
    Path("checkpoints"),
    Path("model/checkpoints"),
    Path("slm-from-scratch/model/checkpoints"),
)
checkpoint_directory = next(
    (
        path
        for path in checkpoint_directory_candidates
        if (path / "latest_checkpoint.pt").exists()
        or (path / "interrupted_checkpoint.pt").exists()
    ),
    Path("checkpoints"),
)
checkpoint_directory.mkdir(
    parents=True,
    exist_ok=True,
)

latest_checkpoint_path = (
    checkpoint_directory / "latest_checkpoint.pt"
)
best_checkpoint_path = (
    checkpoint_directory / "best_checkpoint.pt"
)
interrupted_checkpoint_path = (
    checkpoint_directory / "interrupted_checkpoint.pt"
)

loss_function = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=0.01,
)

best_validation_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0
start_epoch = 0

training_history = {
    "training_loss": [],
    "validation_loss": [],
}


def save_training_checkpoint(
    path,
    resume_epoch,
    batch_number,
):
    torch.save(
        {
            "resume_epoch": resume_epoch,
            "batch_number": batch_number,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "training_history": training_history,
            "best_validation_loss": best_validation_loss,
            "best_epoch": best_epoch,
            "epochs_without_improvement": (
                epochs_without_improvement
            ),
            "model_config": {
                "vocab_size": vocab_size,
                "context_length": context_length,
                "embedding_dim": embedding_dim,
                "num_heads": num_heads,
                "num_layers": num_layers,
            },
        },
        path,
    )


# Select the checkpoint with the greatest training progress.
# Comparing epoch/batch prevents a newer low-progress file
# from replacing an older checkpoint that is farther ahead.
resume_batch = 0
checkpoint_candidates = []

for candidate_path in (
    latest_checkpoint_path,
    interrupted_checkpoint_path,
):
    if candidate_path.exists():
        candidate_checkpoint = torch.load(
            candidate_path,
            map_location=device,
            weights_only=False,
        )
        candidate_progress = (
            candidate_checkpoint["resume_epoch"],
            candidate_checkpoint["batch_number"],
        )
        checkpoint_candidates.append(
            (
                candidate_progress,
                candidate_path,
                candidate_checkpoint,
            )
        )

if checkpoint_candidates:
    _, resume_checkpoint_path, checkpoint = max(
        checkpoint_candidates,
        key=lambda item: item[0],
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )
    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    start_epoch = checkpoint["resume_epoch"]
    resume_batch = checkpoint["batch_number"]
    training_history = checkpoint["training_history"]
    best_validation_loss = checkpoint[
        "best_validation_loss"
    ]
    best_epoch = checkpoint["best_epoch"]
    epochs_without_improvement = checkpoint[
        "epochs_without_improvement"
    ]

    print(
        f"Loaded {resume_checkpoint_path.name}. "
        f"Resuming from epoch {start_epoch + 1}, "
        f"after batch {resume_batch}"
    )
    if resume_batch > 0:
        print(
            "Note: shuffle=True creates a new batch order; "
            "the resume point is approximate for this epoch."
        )
else:
    print("No checkpoint found; starting from scratch.")


interrupted = False
epoch = start_epoch
batch_number = 0

try:
    for epoch in range(start_epoch, num_epochs):
        model.train()

        total_training_loss = 0.0
        total_training_tokens = 0

        epoch_loader = train_loader
        first_batch_number = 1

        if epoch == start_epoch and resume_batch > 0:
            # Skip sampler indices without loading/collating
            # hundreds of thousands of completed batches.
            remaining_batch_sampler = islice(
                train_loader.batch_sampler,
                resume_batch,
                None,
            )
            epoch_loader = DataLoader(
                train_dataset,
                batch_sampler=remaining_batch_sampler,
            )
            first_batch_number = resume_batch + 1
            print(
                f"Fast-forwarded to batch "
                f"{first_batch_number}."
            )

        for batch_number, (input_ids, targets) in enumerate(
            epoch_loader,
            start=first_batch_number,
        ):
            input_ids = input_ids.to(
                device,
                non_blocking=True,
            )
            targets = targets.to(
                device,
                non_blocking=True,
            )

            optimizer.zero_grad(set_to_none=True)

            logits = model(input_ids)

            loss = loss_function(
                logits.reshape(-1, vocab_size),
                targets.reshape(-1),
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite loss detected: {loss.item()}"
                )

            loss.backward()

            gradient_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

            number_of_tokens = targets.numel()

            total_training_loss += (
                loss.item() * number_of_tokens
            )
            total_training_tokens += number_of_tokens

            if batch_number % 100 == 0:
                print(
                    f"Epoch {epoch + 1}/{num_epochs} | "
                    f"Batch {batch_number}/"
                    f"{len(train_loader)} | "
                    f"Loss: {loss.item():.4f} | "
                    f"Gradient norm: "
                    f"{gradient_norm:.4f}"
                )

            if batch_number % checkpoint_frequency == 0:
                # The batch number lets a resumed run skip
                # completed batches. With shuffle=True, the
                # reconstructed order is only approximate.
                save_training_checkpoint(
                    path=latest_checkpoint_path,
                    resume_epoch=epoch,
                    batch_number=batch_number,
                )

                print(
                    f"Checkpoint saved at epoch "
                    f"{epoch + 1}, batch {batch_number}"
                )

        training_loss = (
            total_training_loss
            / total_training_tokens
        )

        validation_loss = evaluate(
            model=model,
            data_loader=validation_loader,
            loss_function=loss_function,
            device=device,
        )

        training_history["training_loss"].append(
            training_loss
        )
        training_history["validation_loss"].append(
            validation_loss
        )

        print(
            f"\nEpoch {epoch + 1}/{num_epochs} completed | "
            f"Train loss: {training_loss:.4f} | "
            f"Validation loss: {validation_loss:.4f}"
        )

        validation_improved = (
            validation_loss
            < best_validation_loss - minimum_improvement
        )

        if validation_improved:
            best_validation_loss = validation_loss
            best_epoch = epoch + 1
            epochs_without_improvement = 0

            save_training_checkpoint(
                path=best_checkpoint_path,
                resume_epoch=epoch + 1,
                batch_number=0,
            )

            print(
                "Validation improved - "
                "best checkpoint saved."
            )

        else:
            epochs_without_improvement += 1

            print(
                "Validation did not improve | "
                f"Patience: {epochs_without_improvement}/"
                f"{early_stopping_patience}"
            )

        # Always save the latest completed epoch.
        save_training_checkpoint(
            path=latest_checkpoint_path,
            resume_epoch=epoch + 1,
            batch_number=0,
        )

        print("Latest epoch checkpoint saved.\n")

        if (
            epochs_without_improvement
            >= early_stopping_patience
        ):
            print(
                f"Early stopping triggered at "
                f"epoch {epoch + 1}."
            )
            break

except KeyboardInterrupt:
    interrupted = True

    save_training_checkpoint(
        path=interrupted_checkpoint_path,
        resume_epoch=epoch,
        batch_number=batch_number,
    )

    print(
        "\nTraining interrupted. Current weights saved to:"
    )
    print(interrupted_checkpoint_path.resolve())


if not interrupted and best_checkpoint_path.exists():
    best_checkpoint = torch.load(
        best_checkpoint_path,
        map_location=device,
        weights_only=False,
    )

    model.load_state_dict(
        best_checkpoint["model_state_dict"]
    )
    optimizer.load_state_dict(
        best_checkpoint["optimizer_state_dict"]
    )

    print(
        f"\nRestored best model from epoch "
        f"{best_epoch} | Best validation loss: "
        f"{best_validation_loss:.4f}"
    )


epochs_completed = len(
    training_history["training_loss"]
)

print("\nTraining session finished")
print("Epochs completed:", epochs_completed)
print("Best epoch:", best_epoch)

if best_validation_loss < float("inf"):
    print(
        "Best validation loss:",
        f"{best_validation_loss:.4f}",
    )
else:
    print("No completed validation epoch yet.")


Loaded interrupted_checkpoint.pt. Resuming from epoch 2, after batch 306
Note: shuffle=True creates a new batch order; the resume point is approximate for this epoch.

Training session finished
Epochs completed: 0
Best epoch: 0
No completed validation epoch yet.


In [64]:
from pathlib import Path


checkpoint_directory = Path("checkpoints")
checkpoint_directory.mkdir(
    parents=True,
    exist_ok=True,
)

checkpoint_path = (
    checkpoint_directory / "slm_checkpoint.pt"
)

final_training_loss = (
    training_history["training_loss"][-1]
    if training_history["training_loss"]
    else None
)

final_validation_loss = (
    best_validation_loss
    if best_validation_loss < float("inf")
    else None
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "model_config": {
            "vocab_size": vocab_size,
            "context_length": context_length,
            "embedding_dim": embedding_dim,
            "num_heads": num_heads,
            "num_layers": num_layers,
        },
        "training_history": training_history,
        "training_loss": final_training_loss,
        "validation_loss": final_validation_loss,
        "epoch": best_epoch,
    },
    checkpoint_path,
)

print(
    f"Final checkpoint saved to: "
    f"{checkpoint_path.resolve()}"
)


Final checkpoint saved to: C:\Users\Tvari\Desktop\TvaritRepo\SLM\slm-from-scratch\model\checkpoints\slm_checkpoint.pt


In [65]:
@torch.no_grad()
def generate_text(
    model,
    prompt,
    merges,
    vocab,
    max_new_tokens=100,
    temperature=1.0,
    top_k=50,
):
    model.eval()

    generated_ids = encode(prompt, merges)

    for _ in range(max_new_tokens):
        model_input = generated_ids[-model.context_length:]

        input_tensor = torch.tensor(
            [model_input],
            dtype=torch.long,
            device=device,
        )

        logits = model(input_tensor)

        next_token_logits = logits[:, -1, :]

        next_token_logits = (
            next_token_logits / temperature
        )

        if top_k is not None:
            k = min(top_k, next_token_logits.size(-1))

            top_values, _ = torch.topk(
                next_token_logits,
                k,
            )

            cutoff = top_values[:, -1].unsqueeze(-1)

            next_token_logits = next_token_logits.masked_fill(
                next_token_logits < cutoff,
                float("-inf"),
            )

        probabilities = torch.softmax(
            next_token_logits,
            dim=-1,
        )

        next_token = torch.multinomial(
            probabilities,
            num_samples=1,
        ).item()

        generated_ids.append(next_token)

    return decode(generated_ids, vocab)

In [66]:
generated_text = generate_text(
    model=model,
    prompt="Once upon a time",
    merges=merges,
    vocab=vocab,
    max_new_tokens=200,
    temperature=0.4,
    top_k=100,
)

print(generated_text)

Once upon a time, a little girl named Lily went to the park to play. She saw a big tree with many flowers. She wanted to climb the tree.

Lily saw a big tree with a red branch on it. She wanted to climb the tree. But the branch was too high for her. She tried to climb the tree, but it didn't work. The tree was still scared, but Lily was still very sad.

Lily was happy to see her branch who could climb the tree. Lily was happy to see her branch on the tree. She learned that sometimes things can be okay.
Once upon a time, there was a little girl named Lily. She loved to play with her toys and her mommy would make her feel better. One day, she went to the park to play with her friends. They played tag and had lots of fun together. Lily had a big heart and her mommy hugged her tight.

After playing for a while, 
